# Генерация обучающих пар «запрос ↔ норма ГЭСН»

Открытая LLM пишет для каждой нормы 5 запросов так, как их сформулировал бы заказчик или прораб.

**Настройки Kaggle (справа, панель Session options):**
- Accelerator: **GPU T4 x2**
- Internet: **On**

Запуск: **Run All**. Результат — файл `synth_raw.jsonl` во вкладке Output (справа). Его нужно скачать и положить в `gesn-search/data/train/`.

In [ ]:
!pip install -q "vllm>=0.6" 2>&1 | tail -2

In [ ]:
import json, re
import pandas as pd

NORMS_URL = "https://raw.githubusercontent.com/malma0/gesn-search/main/data/norms.parquet"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
N_QUERIES = 5

norms = pd.read_parquet(NORMS_URL)
# нормы-добавки («на каждые 5 мм … добавлять») запросами не описываются
norms = norms[~norms["full_name"].str.contains("добавлять|исключать", case=False)].reset_index(drop=True)
print(len(norms), "норм")

In [ ]:
SYSTEM = """Ты помогаешь собрать данные для поиска по строительным сметным нормам (ГЭСН).
Тебе дают официальное описание работы. Придумай, как эту работу назвали бы живые люди, когда ищут её в поиске.
Отвечай только JSON-массивом строк, без пояснений."""

STYLES = [
    "очень коротко, 2–4 слова, как в поисковой строке",
    "как заказчик-неспециалист, бытовыми словами, можно с контекстом (квартира, дача, гараж, баня)",
    "как прораб, профессиональным сленгом",
    "с конкретным параметром из описания (толщина, материал, тип, размер), но своими словами",
    "небрежно: без предлогов, можно с одной опечаткой или сокращением",
]


def build_prompt(row) -> str:
    content = f"\nСостав работ: {row.content}" if row.content else ""
    styles = "\n".join(f"{i + 1}. {s}" for i, s in enumerate(STYLES))
    return f"""Работа: {row.full_name}
Раздел: {row.collection_name}
Единица измерения: {row.unit}{content}

Напиши {N_QUERIES} разных поисковых запроса для этой работы, по одному в каждом стиле:
{styles}

Правила:
- не копируй официальную формулировку, перефразируй;
- не используй слова «устройство», «норма», «ГЭСН» и коды;
- если работа отличается от похожих параметром (толщина, материал, способ), отрази его хотя бы в двух запросах;
- запросы на русском, без кавычек внутри.

Ответ: JSON-массив из {N_QUERIES} строк."""


print(build_prompt(norms.sample(1, random_state=0).iloc[0]))

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, dtype="half", tensor_parallel_size=2, max_model_len=2048, gpu_memory_utilization=0.9)
tok = llm.get_tokenizer()

prompts = [
    tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(r)}],
        tokenize=False, add_generation_prompt=True,
    )
    for r in norms.itertuples()
]
params = SamplingParams(temperature=0.9, top_p=0.95, max_tokens=400, seed=0)
outputs = llm.generate(prompts, params)

In [ ]:
def parse(text: str) -> list[str]:
    match = re.search(r"\[.*\]", text, re.S)
    if not match:
        return []
    try:
        items = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []
    return [s.strip() for s in items if isinstance(s, str) and s.strip()]


bad = 0
with open("/kaggle/working/synth_raw.jsonl", "w", encoding="utf-8") as f:
    for row, out in zip(norms.itertuples(), outputs):
        text = out.outputs[0].text
        queries = parse(text)
        bad += not queries
        f.write(json.dumps({"code": row.code, "queries": queries, "raw": text}, ensure_ascii=False) + "\n")

print(f"готово: {len(norms)} норм, не разобрано ответов: {bad}")
for row, out in list(zip(norms.itertuples(), outputs))[:5]:
    print("\n", row.full_name, "\n  ", parse(out.outputs[0].text))